# Agent 1 — SOP Compliance (QSR Drive-Thru Scenario)

> Verify that drive-thru staff follow standard operating procedures across
> thousands of franchise locations, 24/7. Pattern: retrieve candidate
> moments with semantic search, then VLM-verify each. Same code generalizes
> to any rule × camera scenario — only the search query and the VLM prompt
> change.

## What this notebook shows

- How to call **Visual Search** (`POST /search`) to retrieve candidate
  handoff moments by natural-language description.
- How to call **Visual Intelligence VLM** (`POST /vu/chat/completions`)
  with a strict-JSON prompt to verify each candidate.
- How to aggregate per-clip VLM verdicts into a single compliance summary.

## Endpoints exercised

| Step | Endpoint | What it does |
|---|---|---|
| Index *(optional)* | `POST /upload_url` + `GET /get_metadata` (poll) | Upload + wait for indexing |
| Retrieve | `POST /search` (BY_CLIP) | Find candidate handoff moments |
| Reason | `POST /vu/chat/completions` | VLM verifies each candidate |

The full Visual Agents PRD has eight agents — this is the first scenario
of agent #1 (SOP Compliance). The companion notebook `04_automotive_sop.ipynb`
is the same code with different prompts; together they show how generic the
pattern is.


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


## Helper functions

These wrap the raw HTTP calls so the rest of the notebook reads at the
right altitude. Every call has comments explaining the wire shape and the
gotchas — if you're integrating Memories.ai into your own code, these are
the patterns you'll copy.


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


In [ ]:
def resolve_media_url(video_no):
    """Bridge a videoNo to a public URL the VLM can fetch.

    `/download` streams the raw bytes back to you — it does NOT return a
    hosted URL. To feed a video to /vu/chat/completions you must host it
    yourself. This helper expects either:

      • MEMORIES_MEDIA_URL_TEMPLATE env var (e.g. https://your-cdn/{video_no}.mp4)
      • a MEDIA_URL_MAP dict you populate inline

    If neither is configured, raises so the notebook stops cleanly.
    """
    if video_no in MEDIA_URL_MAP:
        return MEDIA_URL_MAP[video_no]
    tpl = os.environ.get("MEMORIES_MEDIA_URL_TEMPLATE")
    if tpl:
        return tpl.format(video_no=video_no)
    raise RuntimeError(
        f"No media-URL bridge configured for {video_no}. "
        "Set MEMORIES_MEDIA_URL_TEMPLATE or add an entry to MEDIA_URL_MAP."
    )

# Per-notebook overrides: populate this for testing without a CDN.
# A public Memories.ai test asset is included as an example.
MEDIA_URL_MAP = {
    # "VI676024023022092288": "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
}


## Step 1 — point the agent at an indexed video

For this notebook we assume you already have at least one indexed drive-thru
clip in your account. The Visual Search `/search` endpoint will find it as
long as the video has reached `status=PARSE`.

If you need to upload one first, see the optional cell at the bottom of
this notebook. Otherwise replace `VIDEO_NO` below with one of your own.


In [ ]:
# Pick a video to audit. To find one quickly, do a broad search across your
# default namespace and take the first hit.
seed_hits = search("a person", top_k=1, filtering_level=None)
VIDEO_NO  = seed_hits[0]["videoNo"]
UNIQUE_ID = "default"
print(f"Auditing video {VIDEO_NO!r} ({seed_hits[0]['videoName'][:60]}...)")


## Step 2 — retrieve candidate handoff moments

Semantic search returns clips by meaning, not keyword. We cast a wide net
here (`top_k=10`) and let the VLM filter false positives later — running
the VLM blindly across every second of footage would be 100× more
expensive.


In [ ]:
QUERY = "staff physically hands food bag or drinks to customer at drive-thru window"

hits = search(
    QUERY,
    video_nos=[VIDEO_NO],
    unique_id=UNIQUE_ID,
    top_k=10,
    filtering_level="medium",
)
print(f"Got {len(hits)} candidate moments\n")
for h in hits[:5]:
    print(f"  {h['startTime']:>4}s - {h['endTime']:>4}s   score={h['score']:.3f}")


## Step 3 — VLM-verify each candidate

For each candidate we ask Gemini a focused question and force the response
into a strict JSON schema. The schema is the contract between the agent
and downstream aggregation — keep it small and unambiguous.

> Note: The VLM needs a publicly fetchable URL. `/download` streams binary
> bytes — to give Gemini a `file_uri` you must re-host the video on your
> own CDN (or use the public test asset for demos).


In [ ]:
# For a real audit, populate MEDIA_URL_MAP with your own CDN URLs.
# For a demo run, you can map any video_no to the public test asset:
MEDIA_URL_MAP[VIDEO_NO] = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
media_url = resolve_media_url(VIDEO_NO)

SYSTEM = ("You score drive-thru SOP compliance from camera footage. "
          "Be strict — only mark behaviors observed with clear evidence. "
          "Reply strict JSON only, matching the requested schema.")
SCHEMA = ('{"physical_interactions": int, "bag_handed": bool, '
          '"drinks_included": bool, "greeting_observed": bool, "notes": str}')

events = []
for i, h in enumerate(hits):
    start = float(h["startTime"]); end = float(h["endTime"])
    prompt = (
        f"Between {start:.0f}s and {end:.0f}s in this footage, evaluate "
        "drive-thru SOP compliance at the pickup window. Count physical "
        "interactions between staff and customer. Was a food bag handed? "
        "Were drinks included? Did staff greet the customer? "
        f"Reply JSON only, matching: {SCHEMA}"
    )
    raw = vlm_complete(prompt, video_url=media_url, system=SYSTEM)
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        verdict = {"raw": raw, "parse_error": True}
    events.append({"start": start, "end": end, "score": h["score"], "verdict": verdict})
    print(f"[{i+1}/{len(hits)}] {start:.0f}-{end:.0f}s  →  {verdict}")


## Step 4 — aggregate into a compliance summary

A single per-clip JSON object isn't a report yet. The aggregation step is
where the agent's domain logic lives — for SOP compliance that's "how many
candidate moments were fully compliant (bag handed + greeting observed)?".


In [ ]:
compliant = sum(
    1 for e in events
    if isinstance(e["verdict"], dict)
    and e["verdict"].get("bag_handed")
    and e["verdict"].get("greeting_observed")
)

summary = {
    "video_no": VIDEO_NO,
    "candidates_total": len(events),
    "fully_compliant": compliant,
    "events": events,
}
print(f"\n{compliant}/{len(events)} handoff moments fully compliant.\n")
print(json.dumps(summary, indent=2)[:1500])


## Where to go next

- **Generalize to other SOP scenarios**: swap `QUERY` and the VLM prompt.
  `04_automotive_sop.ipynb` is the same flow for an auto-service bay.
- **Wire up your own CDN** so the VLM sees real footage, not the public
  test asset. The MediaURLMap pattern in `resolve_media_url()` is what
  you customize.
- **Run on every shift, every store**: the same code parameterizes over
  `video_nos=[...]` with up to 100 IDs per call.


### Appendix — Index a new video from scratch (optional)

If you don't have a video indexed yet, this cell uploads a public test
video and waits for it to become searchable. Skip if you already have
`VIDEO_NO` from your own library.


In [ ]:
def upload_url(url, *, unique_id="default", camera_model=None, tags=None,
               datetime_taken=None, video_transcription_prompt=None):
    """Visual Search — POST /upload_url. Indexes a video from a public URL.
    Returns {videoNo, videoName, ...}. The video is NOT searchable yet —
    poll wait_for_parse() before /search.
    """
    data = {"url": url, "unique_id": unique_id}
    if camera_model:                 data["camera_model"] = camera_model
    if tags:                         data["tags"] = list(tags)
    if datetime_taken:               data["datetime_taken"] = datetime_taken
    if video_transcription_prompt:   data["video_transcription_prompt"] = video_transcription_prompt
    r = requests.post(f"{VS_HOST}/upload_url", headers=HEADERS, data=data, timeout=120)
    r.raise_for_status()
    envelope = r.json()
    assert envelope.get("code") == "0000", envelope
    return envelope.get("data") or {}


In [ ]:
def get_metadata(video_no):
    """Visual Search — GET /get_metadata. Returns the indexing metadata
    for a video including its `status` (UNPARSE → PARSE)."""
    r = requests.get(f"{VS_HOST}/get_metadata", headers=HEADERS,
                     params={"video_no": video_no}, timeout=30)
    r.raise_for_status()
    envelope = r.json()
    return envelope.get("data")  # can be None right after upload (eventual consistency)


In [ ]:
def wait_for_parse(video_no, *, timeout_sec=1800):
    """Poll get_metadata until status reaches PARSE.

    /upload returns videoStatus=UNPARSE; the video isn't searchable until
    it transitions to PARSE. In production you'd register a `callback`
    URL on /upload to be pushed the transition instead of polling.
    """
    deadline = time.time() + timeout_sec
    delay = 5.0
    last = None
    while time.time() < deadline:
        meta = get_metadata(video_no)
        status = (meta or {}).get("status")
        if status != last:
            print(f"  status -> {status!r}")
            last = status
        if status == "PARSE":
            return meta
        if status in {"FAIL", "FAILED", "ERROR"}:
            raise RuntimeError(f"indexing failed for {video_no}: {meta}")
        time.sleep(delay)
        delay = min(delay * 1.5, 30.0)
    raise TimeoutError(f"indexing did not reach PARSE within {timeout_sec}s")


In [ ]:
# Uncomment to upload the public test asset and wait for indexing.
# up = upload_url(
#     "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
#     unique_id="default",
#     tags=["notebook-smoke"],
#     video_transcription_prompt="Focus on any spoken interactions.",
# )
# print("uploaded:", up)
# wait_for_parse(up["videoNo"])
